# 🔍 CTF Challenge — Găsește flag-ul ascuns în trafic

Ai primit un fișier `.pcap` capturat de pe o rețea.  
Undeva în acel trafic sunt ascunse **4 bucăți dintr-un mesaj secret**.

Misiunea ta: **găsește-le, pune-le în ordine, și asamblează flag-ul.**

---
> **Setup:** Asigură-te că ai `dpkt` instalat:
> ```
> pip install dpkt
> ```


## Pasul 1 — Încarcă pcap-ul și numără pachetele

In [32]:
import dpkt
import socket

# Schimbă calea dacă e nevoie
PCAP_FILE = "challenge.pcap"

packets = []
with open(PCAP_FILE, "rb") as f:
    for ts, buf in dpkt.pcap.Reader(f):
        packets.append((ts, buf))

print(f"Total pachete în pcap: {len(packets)}")


Total pachete în pcap: 34


## Pasul 2 — Inspectează un pachet

Să vedem cum arată un pachet individual.  
Fiecare pachet are layere: **Ethernet → IP → TCP → payload**.


In [33]:
# Uită-te la primul pachet
ts, buf = packets[0]

eth = dpkt.ethernet.Ethernet(buf)
ip  = eth.data
tcp = ip.data

print(f"Timestamp:   {ts:.3f}")
print(f"Src IP:      {socket.inet_ntoa(ip.src)}")
print(f"Dst IP:      {socket.inet_ntoa(ip.dst)}")
print(f"IP ID:       {ip.id}")
print(f"TTL:         {ip.ttl}")
print(f"Src port:    {tcp.sport}")
print(f"Dst port:    {tcp.dport}")
print(f"Payload:     {tcp.data}")


Timestamp:   1773147781.190
Src IP:      23.98.152.197
Dst IP:      113.197.103.48
IP ID:       35393
TTL:         103
Src port:    29899
Dst port:    4524
Payload:     b'XK62q7oKjzeS'


## Pasul 3 — Privire de ansamblu asupra traficului

Hai să vedem toate pachetele dintr-o privire: src IP, dst port, și payload.  
**Observi ceva ciudat?** 🤔


In [34]:
print(f"{'#':<4} {'Src IP':<16} {'Dst port':<10} {'Payload'}")
print("-" * 55)

for i, (ts, buf) in enumerate(packets):
    try:
        eth = dpkt.ethernet.Ethernet(buf)
        ip  = eth.data
        tcp = ip.data
        src = socket.inet_ntoa(ip.src)
        print(f"{i+1:<4} {src:<16} {tcp.dport:<10} {tcp.data}")
    except Exception as e:
        print(f"{i+1:<4} [eroare: {e}]")


#    Src IP           Dst port   Payload
-------------------------------------------------------
1    23.98.152.197    4524       b'XK62q7oKjzeS'
2    33.231.113.101   22805      b'SbIHmmR'
3    79.158.69.230    63168      b'IDF5OJ9Zt'
4    189.18.79.132    48138      b'TWtsJVx'
5    10.0.0.37        9999       b'twork'
6    2.170.120.250    38595      b'mx0IhyJFQb'
7    30.36.68.81      47554      b'JxeAEbUafI'
8    211.157.236.201  44673      b'7Eyn1W3FK'
9    115.117.228.122  12661      b'QYWmil'
10   5.25.10.103      25535      b'0SKb4YCOeZ5BYz'
11   10.0.0.37        9999       b'Compu'
12   114.193.162.72   63118      b'cjAfLKaMnPhlQ'
13   120.167.37.55    17911      b'hKteu05mgOA2p'
14   135.152.95.245   37943      b'7RL888loW9GO'
15   161.109.222.151  24848      b'e4keX7m89tuCr'
16   15.77.243.65     7081       b'PFD8tAdX1PT6V'
17   36.159.174.11    46877      b'KrfNV8AjqTkp5w'
18   214.106.185.215  60401      b'KSwRF710xDKxPk'
19   196.218.149.150  10533      b'JDoEyjKKcX'
20  

## Pasul 4 — Găsește tiparul

Acum că ai văzut traficul, **ce ai observat?**

Scrie codul care filtrează doar pachetele care par "speciale".  
*Hint: uită-te la src IP și la dst port.*


In [35]:
# ✏️  Completează tu aici — filtrează pachetele interesante
suspicious = []

for ts, buf in packets:
    try:
        eth = dpkt.ethernet.Ethernet(buf)
        ip  = eth.data
        tcp = ip.data
        src = socket.inet_ntoa(ip.src)

        # TODO: adaugă condiția ta de filtrare
        if src == "10.0.0.37" and tcp.dport == 9999:
             suspicious.append((tcp.data,ip.id))
        pass

    except:
        pass

print(f"Pachete suspecte găsite: {len(suspicious)}")
for i in suspicious:
    print(i)


Pachete suspecte găsite: 4
(b'twork', 3)
(b'Compu', 1)
(b's2026', 4)
(b'terNe', 2)


## Pasul 5 — Pune bucățile în ordine

Ai găsit pachetele. Dar ordinea lor în fișier e **amestecată intenționat**.  

Există un câmp în header-ul IP care îți spune ordinea corectă.  
*Hint: `ip.id` — ce valori are la pachetele tale suspecte?*


In [49]:
# ✏️  Sortează după câmpul corect și asamblează flag-ul

suspicious.sort(key=lambda x: x[1])
print(suspicious)


flag = b"".join(raspuns for raspuns, _ in suspicious)
print("FLAG:", flag)


[(b'Compu', 1), (b'terNe', 2), (b'twork', 3), (b's2026', 4)]
FLAG: b'ComputerNetworks2026'


## ✅ Flag-ul tău

Dacă ai ajuns aici, scrie flag-ul complet mai jos:


In [51]:

 print(f"🏁 FLAG: {flag}")


🏁 FLAG: b'ComputerNetworks2026'
